# Programming for Data Science — Assignment 05
## Assignment 05: Integrated Project on Computational Linguistic and Linear Regression
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cgpan/hudm5001/blob/main/Assignments/Assignment_05_Integrated_Project.ipynb)

[**Chenguang Pan**](https://cpan.ai/), [**Youmi Lab**](https://youmilab.ai/)
**Teachers College, Columbia University**

This assignment goes with the in-class notebook **`07_Control_Structures_Iterables_Functions.ipynb`**, and it draws on **every session so far**. There is no quiz this time: one project, start to finish, on real student writing.

About this notebook:  
The design by Chenguang Pan, in collaboration with AI. I am responsible for its accuracy.

---
### 📋 How this assignment works (Read me carefully)

- **Run on Google Colab.** Open this notebook with the badge above, then *File ▸ Save a copy in Drive* so you have your own copy to edit.
- **The data.** 3,500 real argumentative essays written by 10th graders in the United States, all answering the same prompt (i.e., the question for the writting task) about driverless cars. Trained raters scored each essay from **1** (weakest) to **6** (strongest), and the file records each writer's **gender** and **race/ethnicity**. The essays come from the **ASAP 2.0** corpus.  
- **The project.** A state testing office asks you three questions:
  1. How are essay **length** and **lexical diversity** (how varied the vocabulary is) related to the score?
  2. How do **girls and boys**, and the **racial/ethnic groups**, differ in score, length, and diversity?
  3. When essay length, lexical diversity, word length, gender, and race/ethnicity go into one **linear regression**, how is each one associated with the score while the others are held constant, and how much of the variation in scores do they account for together?
- **Scope.** Everything builds on **Sessions 01–07**. Unlike the last two assignments, this one *wants* `for` loops, `if` statements, and your own functions (`def`): loops for text, groups, and reports; vectorized pandas for arithmetic on whole columns. `.apply()`, `lambda`, and list comprehensions are fine if you already know them, but nothing here requires them.  
- **Self-check cells** (marked 🔎) run a quick test on your function and print ✅ or ❌. Passing one does not guarantee a perfect answer, and a ❌ tells you where to look. Run them, but do not edit them.
- **AI help.** Please refer to the `Hint` boxes and ask AI for help if you get stuck somewhere or you cannot make sense of the task.
- **Handle with care.** These are real (anonymized) students. Describe what you see; do not guess at causes, and never attribute a difference to the ability, effort, or culture of a group. A gap in averages in this file is not a verdict on any group, and a group label says nothing about any one student.
- Use **clear variable names**, give every function a **docstring** (See Week 7's lecture notebook 3.1 and 3.2), and drop a **comment** on the tricky lines.  

---
### ✍️ Before you start
Put your name in the next cell, and write down any AI tool you used (leave it empty if you used none). Then run the setup cell right after it.

In [2]:
student_name = "Yifei Zhu"   # <-- type your name between the quotes
ai_note = "Help me debug in each of the tasks"        # <-- e.g. "ChatGPT: explained my regression output in Task 5" (leave "" if none)
print("Submission by:", student_name)

Submission by: Yifei Zhu


In [3]:
# === SETUP — run me first! ===
import numpy as np
import pandas as pd
import statsmodels.api as sm   # for the regression in Task 5

# The essays live in the public course repo, so they load straight from the web.
DATA_URL = "https://raw.githubusercontent.com/cgpan/hudm5001/main/asset/asap2_driverless_cars.csv"

essays_raw = pd.read_csv(DATA_URL)

print("essays_raw:", essays_raw.shape)
print("pandas", pd.__version__, "| NumPy", np.__version__)
print("Setup complete ✅")

essays_raw: (3500, 5)
pandas 2.2.3 | NumPy 2.4.6
Setup complete ✅


#### 🗂️ What is in the dataset

| Column | What it holds |
|---|---|
| `essay_id` | an anonymous ID for the essay |
| `score` | the holistic score from trained raters: 1 (weakest) to 6 (strongest) |
| `gender` | `F` or `M`, as recorded by the schools |
| `race_ethnicity` | one of six categories recorded by the schools (coarse labels, and the only ones we have) |
| `full_text` | the essay text as it appears in the corpus, spelling and all |

You will build more columns yourself in Task 3.

---
## Task 1 — Meet the data  

**Difficulty:** ★★☆☆☆
**Skills:** `.shape`, `.columns`, `.head()`, `.iloc`, string slicing, `value_counts()`

**The setting.** Before any analysis you look at what arrived: how big it is, what the columns hold, and what one essay actually looks like.

1. Print the **shape** of `essays_raw` and its **column names** as a list.
2. Show the first 3 rows with `.head(3)`. Notice how long `full_text` is: pandas cuts it off for display.
3. Print just the **first 300 characters** of the first essay. *(Take the `full_text` column, get its first value with `.iloc[0]`, then slice that string with `[:300]`.)*
4. Print how many essays got each score, in score order (`value_counts()`, then `.sort_index()`, which sorts by the labels, here the scores, instead of by the counts).
5. Print how many essays there are for each `gender` and for each `race_ethnicity`.

*(Wrap each item in `print(...)`, so that every output shows up.)*

In [9]:
# Your code here
print(essays_raw.shape)
print(list(essays_raw.columns))
print(essays_raw.head(3))
print(essays_raw['full_text'].iloc[0][:300])
print(essays_raw.value_counts('score').sort_index())
print(essays_raw.value_counts('gender').sort_index())
print(essays_raw.value_counts('race_ethnicity').sort_index())

(3500, 5)
['essay_id', 'score', 'gender', 'race_ethnicity', 'full_text']
               essay_id  score gender          race_ethnicity  \
0  AAAOPP13416000238609      4      M         Hispanic/Latino   
1  AAAOPP13416000238603      2      F  Black/African American   
2  AAAOPP13416000238600      2      F         Hispanic/Latino   

                                           full_text  
0  With technology rapidly evolving, new devices ...  
1  I think that smart cars are absolutly going to...  
2  People think that driveless cars are dangerous...  
With technology rapidly evolving, new devices can be developed and old ones can be improved. However, not all old devices need to be improved further upon. For example, self-driving cars seems to be a catastrophe of an idea. They require manual driving in certain haphazard situations, such as naviga
score
1      74
2     700
3    1352
4    1144
5     220
6      10
Name: count, dtype: int64
gender
F    1833
M    1665
Name: count, dtype: int64


---
## Task 2 — Audit and tidy the table

**Difficulty:** ★★★☆☆
**Skills:** `def`, `return`, `for`, `if`/`elif`/`else`, `continue`, dictionaries, `.dropna()`, `.replace()`

**The setting.** In Session 07 you wrote a loop that lists the columns with missing values. Real projects run that check on every new file, so it deserves to be a function. Then you build a tidy copy of the table that the rest of the project will use.

🧰 **Toolbox**
- `df.dropna(subset=["a", "b"])` drops every row that is missing `a` **or** `b`, and gives you a new table.
- `df["a"].replace({"old": "new"})` swaps values using a dictionary and leaves every other value alone (`.map` would turn them into missing values).
- `df.reset_index(drop=True)` renumbers the rows 0, 1, 2, … after some were dropped.
- `df.copy()` makes an independent copy, so changing it never touches the original.

**Step 1 — Write `audit(df)`**
Loop over `df.columns`. For each column, count the missing values (`.isna().sum()`). If a column has none, skip it with `continue`; otherwise store the count in a dictionary as a plain number, `int(...)`. **Return** the dictionary `{column_name: number_missing}`. Add a docstring. *(This is the loop from Session 07, §2.8, wrapped in a function that returns its result instead of printing it.)*

<details>
<summary>💡 Hint</summary>

Create `problems = {}` **before** the loop and `return problems` **after** it (indented under `def`, not under `for`). Inside the loop: count, `continue` if the count is 0, otherwise `problems[col] = ...`.

</details>

In [23]:
def audit(df):
    """Return a dictionary of columns with missing values and their counts."""
    problems = {}
    for i in list(df.columns):
        value = df[i].isna().sum()
        if value == 0:
            continue
        else:
            problems[i] = int(value)
    return problems

In [24]:
# 🔎 Self-check — run me, do not edit
toy = pd.DataFrame({"a": [1, None, 3], "b": [1, 2, 3], "c": [None, None, "x"]})
if audit(toy) == {"a": 1, "c": 2}:
    print("✅ audit works on the toy table")
else:
    print("❌ audit returned", audit(toy), "but should return {'a': 1, 'c': 2}")

✅ audit works on the toy table


**Step 2 — Write `severity(pct)`**
Take a percentage of missing values and return a label from an `if`/`elif`/`else` ladder: under 1 → `"negligible"`, under 5 → `"minor"`, under 30 → `"moderate"`, otherwise `"severe"`. Add a docstring.

In [25]:
def severity(pct):
  """Return a severity label based on the percentage of missing values."""
  if pct < 1:
    return('negligible')
  elif pct < 5:
    return('minor')
  elif pct < 30:
    return('moderate')
  else:
    return('severe')

In [26]:
# 🔎 Self-check — run me, do not edit
checks = [(0.2, "negligible"), (1, "minor"), (3, "minor"), (5, "moderate"), (12, "moderate"), (30, "severe"), (40, "severe")]
all_good = True
for pct, expected in checks:
    if severity(pct) != expected:
        all_good = False
        print("❌ severity(", pct, ") gave", severity(pct), "but should give", expected)
if all_good:
    print("✅ severity works")

✅ severity works


**Step 3 — Audit the real table**
Run `audit(essays_raw)` and store the result as `problems`. Then loop over `problems.items()` and print one line per column, like `gender: 2 missing (0.06%), negligible`. *(Percent = missing ÷ number of rows × 100, rounded to 2 decimals. Call `severity()` for the label.)*

In [29]:
# Your code here
problems=audit(essays_raw)
for k,v in problems.items():
  pct=v/essays_raw[k].shape[0]*100
  sev=severity(pct)
  print(f'{k}:{v} missing ({pct:.2f}%),{sev}')

gender:2 missing (0.06%),negligible
race_ethnicity:2 missing (0.06%),negligible


**Step 4 — Write `clean_essays(raw)`**
Given the raw table, return a **tidy copy** (never change `raw` itself). The function should:
1. drop the rows that are missing `gender` or `race_ethnicity`, however you like (`dropna` or a boolean mask), and end that line with `.copy()` so you can safely change the result;
2. shorten the labels with the dictionary below: loop over `LABEL_MAPS.items()` (column name, mapping) and replace the values in each of those columns;  
   >**Tip:** people sometimes call the categories "labels" or "levels" in a categorical column. Some category's name is too long, we want to change it into a shorter one. If you do not understand what this question means, no worries: just ask AI for help.
3. renumber the rows with `reset_index(drop=True)` (Task 3 glues new columns on by row number, so the numbers must run 0, 1, 2, … without gaps), and return the table. Add a docstring.

Run the next cell first: it defines the label shortcuts. A function can read a variable defined earlier in the notebook (here `LABEL_MAPS`), so you do not need to pass it in.

In [30]:
# Run me: the label shortcuts that clean_essays() should apply
LABEL_MAPS = {
    "gender": {"F": "Female", "M": "Male"},
    "race_ethnicity": {
        "Asian/Pacific Islander": "Asian/PI",
        "American Indian/Alaskan Native": "Am. Indian/AK Native",
        "Two or more races/Other": "Multiracial/Other",
    },
}

<details>
<summary>💡 See hint or use AI for this task if get stuck</summary>

The pattern for one column is `tidy[column] = tidy[column].replace(mapping)`. Put it inside `for column, mapping in LABEL_MAPS.items():`. If pandas prints a `SettingWithCopyWarning`, you are changing a slice of another table: end the line that drops the rows with `.copy()`.  

Please ask AI for help if you do not understand what this question means. After that, please think about what you've learned about revising the value labels (categorical levels) .

</details>

In [36]:
def clean_essays(raw):
  cleaned_raw=raw.dropna(subset=["gender", "race_ethnicity"]).copy()
  for k,v in LABEL_MAPS.items():
        if k in list(cleaned_raw.columns):
            cleaned_raw[k] = cleaned_raw[k].replace(v)
  return cleaned_raw.reset_index(drop=True)

**Step 5 — Build `clean` and check it**
Run `clean = clean_essays(essays_raw)`. Print the shape of the raw and the clean table, `audit(clean)` (it should come back empty), the number of duplicated `essay_id`s (`.duplicated().sum()`), and the `value_counts()` of the two group columns `gender`, `race_ethnicty`to see the new labels.

In [41]:
# Your code here
clean = clean_essays(essays_raw)
print(clean.shape,clean)
print(audit(clean))
print(clean['essay_id'].duplicated().sum())
print(clean.value_counts('gender'),clean.value_counts('race_ethnicity'))

(3498, 5)                   essay_id  score  gender          race_ethnicity  \
0     AAAOPP13416000238609      4    Male         Hispanic/Latino   
1     AAAOPP13416000238603      2  Female  Black/African American   
2     AAAOPP13416000238600      2  Female         Hispanic/Latino   
3     AAAOPP13416000238592      2  Female  Black/African American   
4     AAAOPP13416000238583      2  Female  Black/African American   
...                    ...    ...     ...                     ...   
3493  AAAOPP13416000000088      4    Male                   White   
3494  AAAOPP13416000000071      4  Female  Black/African American   
3495  AAAOPP13416000000045      3    Male  Black/African American   
3496  AAAOPP13416000000022      4  Female                   White   
3497  AAAOPP13416000000015      3    Male                   White   

                                              full_text  
0     With technology rapidly evolving, new devices ...  
1     I think that smart cars are absolutly g

In [42]:
# 🔎 Self-check — run me, do not edit
test_clean = clean_essays(essays_raw)
if test_clean is None:
    print("❌ clean_essays returned nothing: did you forget `return`?")
else:
    short_race = {"Asian/PI", "Am. Indian/AK Native", "Black/African American", "Hispanic/Latino", "Multiracial/Other", "White"}
    checks = {
        "2 unlabeled rows dropped": test_clean.shape[0] == essays_raw.shape[0] - 2,
        "raw table untouched": essays_raw.shape[0] == 3500 and set(essays_raw["gender"].dropna()) == {"F", "M"},
        "labels shortened": set(test_clean["gender"]) == {"Female", "Male"} and set(test_clean["race_ethnicity"]) == short_race,
        "no missing values left": audit(test_clean) == {},
    }
    for name, ok in checks.items():
        print("✅" if ok else "❌", name)
    if not checks["raw table untouched"]:
        print("   Your function changed essays_raw. Fix it, re-run the Setup cell to reload the data, then run this check again.")

✅ 2 unlabeled rows dropped
✅ raw table untouched
✅ labels shortened
✅ no missing values left


---
## Task 3 — Turn text into numbers

**Difficulty:** ★★★★☆
**Skills:** string methods, `for` over characters, `if`, `continue`, `set`, dictionaries, `def`/`return`, accumulators, building a list in a loop, `pd.DataFrame`, `pd.concat`, `.corr()`

**The setting.** A computer cannot read an essay, but it can count. You will turn each essay into a few numbers, and everything after this task works on those numbers.

- A **token** is one word as written (if "cars" appears three times, that is three tokens).
- A **type** is one *distinct* word (those three tokens are one type).
- The **type-token ratio (TTR)** is types ÷ tokens. It is a classic measure of lexical diversity: 1.0 means no word is ever repeated.
- **The catch:** TTR shrinks as an essay gets longer, because long essays reuse common words ("the", "and", "cars"). A long essay looks "less diverse" just for being long. To compare essays fairly we also compute **`ttr100`**, the TTR of only the **first 100 tokens**, so every essay is judged on the same number of words. You will check this claim with data in Step 4.
- **Average word length** (characters per token) is a rough sign of how sophisticated the vocabulary is.
- *Type* and *token* are linguistics words here: a type is not a Python data type, and a token is just a word.

🧰 **Toolbox**
- `ch.isalpha()` is `True` when a character (or a string) is made only of letters.
- `text.lower()`, `text.replace(old, new)`, `text.split()` (splits on whitespace), and `word.strip("'")` (removes apostrophes from both ends) are string methods like the ones from Session 02.
- `len(set(words))` counts the distinct items in a list.
- `for text in clean["full_text"]:` loops over a column's **values**, one essay at a time.
- `pd.DataFrame(list_of_dicts)` turns a list of dictionaries into a table (one row per dictionary), and `pd.concat([a, b], axis=1)` puts two tables side by side.

**Step 1 — Write `tokenize(text)`**
Return the essay as a list of words. Follow these rules exactly, so that everyone gets the same counts:
1. lowercase everything;
2. replace every character in `ODD_APOSTROPHES` with a normal apostrophe `'` (in a few essays the apostrophe shows up as `´` or as the stray character `\x92`, an encoding leftover);
3. go through the text **one character at a time**: keep letters and apostrophes, and turn everything else (digits, punctuation, line breaks, dashes) into a space. Build a new string as you go;
4. split the new string on whitespace;
5. strip apostrophes from both ends of each piece, and skip any piece that ends up empty (use `continue`).

Run the next cell first: it defines the apostrophe look-alikes.

In [ ]:
# Run me: apostrophe look-alikes that appear in a few essays
ODD_APOSTROPHES = ["’", "‘", "´", "\x92"]

<details>
<summary>💡 See hint or use AI if get stuck</summary>

Build the cleaned string with an accumulator, just like `total += s`, but for text:

```python
cleaned = ""
for ch in text:
    if ch.isalpha() or ch == "'":
        cleaned += ch
    else:
        cleaned += " "
```

Then `cleaned.split()` gives the pieces to strip and collect in a list.

</details>

In [ ]:
def tokenize(text):
    pass   # replace this line with your code (start with a docstring!)

In [ ]:
# 🔎 Self-check — run me, do not edit
test = tokenize("Don´t STOP -- it's 2 cars!")
if test == ["don't", "stop", "it's", "cars"]:
    print("✅ tokenize works on the first toy sentence")
else:
    print("❌ tokenize returned", test, "but should return", ["don't", "stop", "it's", "cars"])

test2 = tokenize("'Quoted' words ' it’s fine")
if test2 == ["quoted", "words", "it's", "fine"]:
    print("✅ tokenize works on the second toy sentence")
else:
    print("❌ tokenize returned", test2, "but should return", ["quoted", "words", "it's", "fine"])

**Step 2 — Write `lexical_stats(tokens)`**
Take a list of tokens and return **one dictionary** with five keys:

| key | meaning |
|---|---|
| `"n_tokens"` | how many tokens |
| `"n_types"` | how many distinct tokens |
| `"ttr"` | `n_types / n_tokens` |
| `"ttr100"` | the number of distinct tokens among the **first 100**, divided by the number of tokens in that first-100 slice |
| `"avg_word_len"` | total number of characters summed across all tokens ÷ number of tokens, with the total found by an **accumulator loop** (`total += len(word)`) |

Do not round inside the function; round only when you print. Add a docstring.

<details>
<summary>💡 Hint</summary>

`first_100 = tokens[:100]` gives the first 100 tokens. Divide by `len(first_100)` rather than by 100, so the function would also work on a short essay.

</details>

In [ ]:
def lexical_stats(tokens):
    """Get lexical stats from a token list created by the upstream function `tokenize`"""
    pass   # replace this line with your code (start with a docstring!)

In [ ]:
# 🔎 Self-check — run me, do not edit
stats = lexical_stats(["the", "cat", "the", "dog"])
expected = {"n_tokens": 4, "n_types": 3, "ttr": 0.75, "ttr100": 0.75, "avg_word_len": 3.0}
if stats == expected:
    print("✅ lexical_stats works on the short toy list")
else:
    print("❌ lexical_stats returned", stats, "but should return", expected)

many = []
for i in range(100):
    many.append("w" + str(i))        # 100 different words ...
many = many + ["w0"] * 50            # ... then 50 repeats of one word: 150 tokens in all
long_stats = lexical_stats(many)
if (long_stats is not None and long_stats["n_tokens"] == 150 and long_stats["n_types"] == 100
        and long_stats["ttr100"] == 1.0 and abs(long_stats["ttr"] - 2 / 3) < 1e-9):
    print("✅ lexical_stats works on the long toy list (ttr100 looks at the first 100 tokens only)")
else:
    print("❌ on a 150-token list, ttr100 should be 1.0 and ttr should be 0.667; got", long_stats)

**Step 3 — Measure every essay**
**Start the cell with `clean = clean_essays(essays_raw)`**, so that running it twice never piles up duplicate columns. Then:
1. Make an empty list `stats_rows`. Loop over `clean["full_text"]`: for each essay, `tokenize` it, call `lexical_stats` on the tokens, and `append` the resulting dictionary to `stats_rows`. (Each dictionary looks like `{"n_tokens": 412, "n_types": 201, ...}`, so the finished list holds one dictionary per essay.)
2. Turn `stats_rows` into a table with `pd.DataFrame(...)` and put it next to `clean` with `pd.concat(..., axis=1)`. Keep the result in the name `clean`.
3. Print `clean.shape` (it should read `(3498, 10)`: the five original columns plus five new ones) and `clean[["n_tokens", "ttr", "ttr100", "avg_word_len"]].describe().round(2)`.

<details>
<summary>💡 Hint</summary>

Without that first line, a second run of this cell would glue the five new columns onto a table that already has them, and later steps would break in confusing ways. (Forgetting `axis=1` is the other classic slip: you would end up with 6,996 rows.)

</details>

In [ ]:
# Your code here

In [ ]:
# 🔎 Self-check — run me, do not edit
new_cols = ["n_tokens", "n_types", "ttr", "ttr100", "avg_word_len"]
if "clean" not in globals() or not all(col in clean.columns for col in new_cols):
    print("❌ the five new columns are not in `clean` yet: finish Step 3 first")
else:
    checks = {
        "shape is (3498, 10)": clean.shape == (3498, 10),
        "column names are unique": bool(clean.columns.is_unique),
        "no missing values in the new columns": int(clean[new_cols].isna().sum().sum()) == 0,
        "average essay length is about 393 tokens": abs(float(clean["n_tokens"].mean()) - 393.2) < 1,
    }
    for name, ok in checks.items():
        print("✅" if ok else "❌", name)

**Step 4 — Is raw TTR (i.e., lexical diversity) positively or negetively related to writing performance (score) or length of writting sample (n_tokens)? Aha, someting interesting!**
Print the correlation table (rounded to 2 decimals) for `score`, `n_tokens`, `ttr`, and `ttr100`. Look at the `ttr` row and the `ttr100` row: what do they say about length and about the score?

In [ ]:
# Your code here

---
## Task 4 — Group differences

**Difficulty:** ★★★☆☆
**Skills:** `def` with parameters, `groupby`, `.items()`, f-strings, conditional expression, `continue`, best-so-far tracker

**The setting.** Do girls and boys, or the racial/ethnic groups, differ in score, essay length, or lexical diversity? You will find out by writing **one reusable report function** and calling it twice. Keep in mind that a difference in averages describes the groups *in this file*; it does not say **why** the difference exists.

**Step 1 — Write `group_report(df, column)`**
It takes a table and the **name of a grouping column** (for example `"gender"`), prints **one line per group**, and returns a dictionary `{group: mean score}`.
- Get the mean `score`, `n_tokens`, and `ttr100` for each group with `df.groupby(column)[[...]].mean()`, and the group sizes with `df[column].value_counts()`.
- Loop over the sizes with `.items()`. Each line shows the group, its size `n`, and the three means (the score rounded to 2 decimals, the word count as a whole number with `round(x)`, and `ttr100` rounded to 3), for example `Example | n=120 | score 3.3 | words 403 | ttr100 0.661`.
- A group with **fewer than 50 essays** gets a warning at the end of its line, added with a conditional expression: `"  <- small group, read with care" if n < 50 else ""`.
- Store each group's mean score in the dictionary as a plain `float(...)` (NumPy numbers print as `np.float64(3.29…)` inside a dictionary, and `float()` turns them into ordinary numbers), and return it. Add a docstring.

<details>
<summary>💡 Hint</summary>

```python
means = df.groupby(column)[["score", "n_tokens", "ttr100"]].mean()
sizes = df[column].value_counts()
for group, n in sizes.items():
    # means.loc[group, "score"] is the mean score of this group
    ...
```

</details>

In [ ]:
# I feel this question is too complex. I provide this function for you.
# Please read this function line by line and make sense of it.

def group_report(df, column):
    """Print one summary line per group in `column`; return {group: mean score}."""
    means = df.groupby(column)[["score", "n_tokens", "ttr100"]].mean()   # one row per group
    sizes = df[column].value_counts()                                    # group -> number of essays

    mean_scores = {}
    for group, n in sizes.items():
        score = means.loc[group, "score"]
        words = means.loc[group, "n_tokens"]
        diversity = means.loc[group, "ttr100"]
        warning = "  <- small group, read with care" if n < 50 else ""   # conditional expression
        print(f"{group} | n={n} | score {round(score, 2)} | words {round(words)} | ttr100 {round(diversity, 3)}{warning}")
        mean_scores[group] = float(score)                                # plain float, not np.float64
    return mean_scores

In [ ]:
# 🔎 Self-check — run me, do not edit (it prints two report lines first)
toy = pd.DataFrame({"g": ["a", "a", "b"], "score": [1, 3, 5], "n_tokens": [100, 200, 300], "ttr100": [0.5, 0.6, 0.7]})
out = group_report(toy, "g")
if out is not None and out == {"a": 2.0, "b": 5.0} and all(type(v) is float for v in out.values()):
    print("✅ group_report works on the toy table")
else:
    print("❌ group_report returned", out, "but should return {'a': 2.0, 'b': 5.0} (plain floats)")

**Step 2 — Run the report twice**
Call `group_report(clean, "gender")` and store the result as `gender_means`. Call `group_report(clean, "race_ethnicity")` and store it as `race_means`. Print both dictionaries too.

In [ ]:
# Your code here

**Step 3 — Highest and lowest**
Use the **best-so-far pattern** from Session 07 on `race_means`: one loop that finds the group with the **highest** mean score and the group with the **lowest**. **Skip groups with fewer than 50 essays** with `continue` (their averages are too shaky to rank; look the sizes up with `clean["race_ethnicity"].value_counts()`). Finish with one f-string such as `Highest: ... (score), lowest: ... (score), gap: ... points` (scores rounded to 2 decimals), and add one `#` comment saying what this gap does **not** tell us.

<details>
<summary>💡 Hint</summary>

A lowest-so-far tracker has to start **high**: scores never exceed 6, so `lowest_score = 99` works (starting at 0 would never be beaten). Inside the loop use two separate `if` statements, not `elif`, so that the very first group can set both trackers.

</details>

In [ ]:
# Your code here

---
## Task 5 — Guided practice: fit a multiple linear regression  

**Difficulty:** ★★★★★ (guided)
**Skills:** dummy coding with `pd.Categorical` and `pd.get_dummies`, `pd.concat`, statsmodels (`sm.add_constant`, `sm.OLS(...).fit()`, `.summary()`), reading a regression summary

**The research question.** How are essay **length**, **lexical diversity**, **word length**, **gender**, and **race/ethnicity** associated with the essay **score** when all five are considered together?  

**How this task works.** The task walks you through the analysis in two steps: dummy-code the groups, then fit the model and read its summary. Most of the code is written for you. In each code cell, **replace every `____`** (four underscores) with the right code, run the cell, then run the 🔎 self-check under it. A cell that still contains a `____` stops with an error that mentions `____`. If regression is new to you, read the 📐 cell below slowly, and feel free to ask an AI tool to explain any part of it.

#### 📐 The model

In words: *score = a starting value + a slope for each text feature + a shift for each group + whatever the formula misses.* In symbols, for essay $i$ (one of the $n$ = 3,498 essays):

$$
\begin{aligned}
\text{score}_i = \beta_0 &+ \beta_1\,\text{length}_i + \beta_2\,\text{diversity}_i + \beta_3\,\text{wordlen}_i \\
&+ \beta_4\,\text{Male}_i \\
&+ \beta_5\,\text{Black}_i + \beta_6\,\text{Hispanic}_i + \beta_7\,\text{Multiracial}_i + \beta_8\,\text{AsianPI}_i + \beta_9\,\text{AmIndian}_i \\
&+ \varepsilon_i
\end{aligned}
$$

| In the formula | Name in the code | What it is |
|---|---|---|
| $\text{score}_i$ | `score` | the outcome: the essay's score, 1 to 6 |
| $\text{length}_i$ | `words_100` | essay length in **hundreds** of words (`n_tokens / 100`, so 3.9 means 390 words) |
| $\text{diversity}_i$ | `ttr100` | lexical diversity: the type-token ratio of the first 100 words (0 to 1) |
| $\text{wordlen}_i$ | `avg_word_len` | average word length, in characters |
| $\text{Male}_i$ | `Male` | 1 if the writer is recorded as male, 0 if female |
| $\text{Black}_i$, $\text{Hispanic}_i$, $\text{Multiracial}_i$, $\text{AsianPI}_i$, $\text{AmIndian}_i$ | `Black/African American`, `Hispanic/Latino`, `Multiracial/Other`, `Asian/PI`, `Am. Indian/AK Native` | 1 if the writer is in that race/ethnicity group, 0 otherwise |
| $\beta_0$ | `const` | the intercept |
| $\beta_1, \dots, \beta_9$ | – | the coefficients: what the regression estimates |
| $\varepsilon_i$ | – | the error: the part of the essay's score that the formula does not capture |

**Reading the coefficients.** Each $\beta$ describes one predictor **with all the other predictors held constant**.
- $\beta_1$, $\beta_2$, $\beta_3$: the change in the predicted score when that predictor goes up by 1 unit. One unit is 100 words for $\beta_1$, the whole 0-to-1 range of `ttr100` for $\beta_2$ (far more than essays really differ: the middle half of them lie between 0.63 and 0.70), and one more character per word, on average, for $\beta_3$. The units differ, so the sizes of these three coefficients cannot be compared with each other.
- $\beta_4$ to $\beta_9$: the difference in predicted score between a group and its **reference group**: male vs. female writers for $\beta_4$, and each group vs. White writers for $\beta_5$ to $\beta_9$. Step 1 explains reference groups.
- $\beta_0$: the predicted score when every predictor is 0 (an essay with no words, by a writer in both reference groups). No real essay looks like that, so $\beta_0$ only anchors the formula; do not interpret it.

**What the regression does.** It estimates every $\beta$ from the data (an estimate wears a hat: $\hat\beta$). **Ordinary least squares (OLS)** picks the $\hat\beta$'s that make the sum of the squared residuals (actual minus predicted score) as small as possible:

$$
\widehat{\text{score}}_i = \hat\beta_0 + \hat\beta_1\,\text{length}_i + \dots + \hat\beta_9\,\text{AmIndian}_i,
\qquad \text{OLS minimizes } \sum_{i=1}^{n} \left(\text{score}_i - \widehat{\text{score}}_i\right)^2 .
$$

The t tests and p values you will get assume that the errors are independent and roughly normal, with the same spread for every essay: $\varepsilon_i \sim N(0, \sigma^2)$.

Keep two things in mind. A coefficient describes an **association** in these essays, not a cause. And because we fit and judge the model on the same essays, its R² describes how well it fits **these** essays, not how well it would predict new ones.

**Step 1 — Dummy-code gender and race/ethnicity**

A regression can only multiply numbers, but `gender` and `race_ethnicity` hold labels such as `"Male"` or `"Hispanic/Latino"`. **Dummy coding** turns a column of group labels into columns of 0s and 1s, one for every group but one:
- a **dummy** is **1** when the essay's writer is in that group and **0** otherwise;
- one group gets **no** dummy: the **reference group**. Its essays are 0 in every dummy, and each dummy's coefficient compares its group **with the reference group**.

Race/ethnicity has 6 groups. With White as the reference group, it needs 5 dummies, each named after its group:

| `race_ethnicity` | Black/African American | Hispanic/Latino | Multiracial/Other | Asian/PI | Am. Indian/AK Native |
|---|:---:|:---:|:---:|:---:|:---:|
| White *(reference)* | 0 | 0 | 0 | 0 | 0 |
| Black/African American | 1 | 0 | 0 | 0 | 0 |
| Hispanic/Latino | 0 | 1 | 0 | 0 | 0 |
| Multiracial/Other | 0 | 0 | 1 | 0 | 0 |
| Asian/PI | 0 | 0 | 0 | 1 | 0 |
| Am. Indian/AK Native | 0 | 0 | 0 | 0 | 1 |

Gender has 2 groups. With Female as the reference group, it needs 1 dummy, `Male`.

- **Why leave one group out?** With a dummy for every group, the dummies would add up to 1 in every row, exactly like the column of 1s behind the intercept, and the model could not tell them apart (the *dummy variable trap*).
- **Why Female and White?** They are the largest groups in the file (1,833 and 1,179 essays), so comparisons with them are the most precise. This is a statistical convenience, not a claim that any group is the standard. Another choice would change what each dummy compares, but not the model's predictions or its R².

🧰 **Toolbox — dummies from a categorical**

A **categorical** is pandas' type for a column of group labels. It keeps the list of groups, its **categories**, in an order you choose. `pd.get_dummies` makes one 0/1 column per category, in that order, and `drop_first=True` leaves out the **first** one. So list the reference group first, and it is the group left out.

```python
colors = pd.Categorical(df["color"], categories=["red", "green", "blue"])   # "red" first: the reference group
color_dummies = pd.get_dummies(colors, drop_first=True, dtype=int)         # 0/1 columns "green" and "blue", no "red"
```

`dtype=int` gives 1s and 0s instead of `True`/`False`, which statsmodels refuses. Under the hood, each dummy is a comparison turned into numbers: the `green` column equals `(df["color"] == "green").astype(int)`.

**Careful:** without `categories=`, pandas sorts the categories alphabetically, so `drop_first=True` leaves out whichever group comes first in the alphabet. Here that is the 34-essay Am. Indian/AK Native group, a poor reference group, and White would get a dummy instead. Always check which group was left out.

**Your job.** Run the next cell, which lists each column's groups with the reference group first. Then complete the cell after it:
1. `model_df` starts as a copy of the columns the model needs (given). Add `words_100`, the length in hundreds of words.
2. The gender lines are written for you. Write the matching race/ethnicity lines: a categorical of `clean["race_ethnicity"]` with the categories in `CATEGORIES["race_ethnicity"]`, then its dummies.
3. Run the cell. Check that White is not among the dummies, and compare the counts with Task 1: 1,665 male writers, 849 Black/African American writers, and so on.

<details>
<summary>💡 Hint</summary>

Copy each gender line and change what makes it about gender: the variable names, the column `clean["gender"]`, and the key `"gender"` in `CATEGORIES`. If White shows up among the dummies, the categories did not reach `pd.Categorical`.

</details>

In [ ]:
# Run me: the groups in each label column, with the REFERENCE GROUP FIRST
# (Female and White, the largest groups: pd.get_dummies(..., drop_first=True) will leave them out)
CATEGORIES = {
    "gender": ["Female", "Male"],
    "race_ethnicity": ["White", "Black/African American", "Hispanic/Latino",
                       "Multiracial/Other", "Asian/PI", "Am. Indian/AK Native"],
}

In [ ]:
# Step 1 — build the table for the model. Fill in each blank, then run.
model_df = clean[["score", "n_tokens", "ttr100", "avg_word_len"]].copy()   # .copy(), so `clean` stays as it is

# (a) create a new column `words_100` to indicate essay length in hundreds of words: n_tokens / 100
model_df["words_100"] = ____

# (b) the label columns as categoricals: the first category is the reference group
gender = pd.Categorical(clean["gender"], categories=CATEGORIES["gender"])
# do the same for race_ethnicity
race = ____

# (c) one 0/1 dummy per category; drop_first=True leaves out the first one, the reference group
gender_dummies = pd.get_dummies(gender, drop_first=True, dtype=int)
# do the same for race
race_dummies = ____

# (d) glue the dummies on next to the other columns,using pd.concat() row by row (axis=1, as in Task 3)
model_df = ____

# (e) look at the result
print("race/ethnicity categories:", list(race.categories))   # the first one is the reference group
print(model_df.head())
print("\nEssays coded 1 in each dummy:")
print(model_df[list(gender_dummies.columns) + list(race_dummies.columns)].sum())

In [ ]:
# 🔎 Self-check — run me, do not edit
expected_dummies = ["Male", "Black/African American", "Hispanic/Latino", "Multiracial/Other", "Asian/PI", "Am. Indian/AK Native"]
if "model_df" not in globals() or "words_100" not in model_df.columns:
    print("❌ model_df or its words_100 column is missing: finish Step 1 first")
else:
    found = [d for d in expected_dummies if d in model_df.columns]
    label_column = {d: "gender" if d == "Male" else "race_ethnicity" for d in expected_dummies}   # the column behind each dummy
    same_rows = len(model_df) == len(clean)
    checks = {
        "one row per essay (3,498)": len(model_df) == 3498,
        "words_100 is n_tokens / 100": same_rows and bool(np.allclose(model_df["words_100"], clean["n_tokens"] / 100)),
        "a dummy for each of the 6 other groups": len(found) == 6,
        "no dummy for the reference groups (Female, White)": "Female" not in model_df.columns and "White" not in model_df.columns,
        "dummies are 0/1 numbers, not True/False": len(found) > 0 and all(not pd.api.types.is_bool_dtype(model_df[d]) and set(model_df[d].unique()) <= {0, 1} for d in found),
        "each dummy is 1 for exactly its own group's essays": same_rows and len(found) == 6 and all(bool((model_df[d].to_numpy() == (clean[label_column[d]] == d).to_numpy()).all()) for d in found),
    }
    for name, ok in checks.items():
        print("✅" if ok else "❌", name)

**Step 2 — Fit the model and print the summary**

We fit the model with **statsmodels**, which reports the full set of regression statistics: coefficients, standard errors, t values, p values, confidence intervals, R², and more. (scikit-learn, the other popular choice, gives the coefficients but no t or p values.)

🧰 **Toolbox — the statsmodels recipe**

```python
import statsmodels.api as sm                  # done in the Setup cell

X = sm.add_constant(df[["col_a", "col_b"]])   # the predictors as a table, plus a column of 1s named "const" (for β0)
y = df["outcome"]                             # the outcome

results = sm.OLS(y, X).fit()                  # OLS = ordinary least squares. Careful: y comes FIRST, then X
print(results.summary())                      # the full regression table
```

Complete the next cell:
1. Build `PREDICTORS`, the nine columns on the right-hand side of the formula, in the formula's order. Two lists are ready for you: `TEXT_FEATURES`, and `DUMMIES`, the names of the dummies Step 1 made (`Male`, then the five race/ethnicity groups).
2. Fill in `y` and the two inputs of `sm.OLS`. The fitted model is stored as `ols_model`.
3. Run the cell. The 🔍 cell after the self-check shows how to read the summary.

<details>
<summary>💡 Hint</summary>

- Two lists join into one with `+`: `[1, 2] + [3]` gives `[1, 2, 3]`. The text features come first, as in the formula.
- `ValueError: Pandas data cast to numpy dtype of object` means a dummy holds `True`/`False` instead of 1/0. Go back to Step 1 and pass `dtype=int` to `pd.get_dummies`.
- `ValueError: shapes (3498,10) and (3498,10) not aligned` means the inputs of `sm.OLS` are swapped: the outcome comes first, `sm.OLS(y, X)`.

</details>

In [ ]:
# We imported statsmodels in the Setup cell. Just in case you skipped it, we import it again here.
import statsmodels.api as sm

# Step 2 — fit the model. Fill in each blank, then run.
TEXT_FEATURES = ["words_100", "ttr100", "avg_word_len"]
DUMMIES = list(gender_dummies.columns) + list(race_dummies.columns)   # the dummy names from Step 1
# glue all list of predictors names together as PREDICTORS:
PREDICTORS = ____

X = sm.add_constant(model_df[PREDICTORS])   # the nine predictors, plus a column of 1s named "const" (for β0)
y = model_df[____]                          # the outcome

ols_model = sm.OLS(____, ____).fit()        # outcome first, then predictors
print(ols_model.summary())

In [ ]:
# 🔎 Self-check — run me, do not edit
expected_terms = ["const", "words_100", "ttr100", "avg_word_len", "Male", "Black/African American",
                  "Hispanic/Latino", "Multiracial/Other", "Asian/PI", "Am. Indian/AK Native"]
if "ols_model" not in globals():
    print("❌ there is no ols_model yet: finish Step 2 first")
else:
    terms = list(ols_model.params.index)
    checks = {
        "the outcome is score": ols_model.model.endog_names == "score",
        "the model has an intercept (const)": "const" in terms,
        "all nine predictors are in the model": set(terms) == set(expected_terms),
        "the predictors are in the formula's order": terms == expected_terms,
        "all 3,498 essays were used": int(ols_model.nobs) == 3498,
    }
    for name, ok in checks.items():
        print("✅" if ok else "❌", name)

#### 🔍 How to read the summary

The summary has three blocks. Find each one in your output.

- **Top block: the whole model.** `Dep. Variable` is the outcome (`score`). `No. Observations` is *n*, the number of essays (3,498). `Df Model` is *k*, the number of predictors (9), and `Df Residuals` is *n* − *k* − 1 (3,488). `R-squared`, `Adj. R-squared`, `F-statistic`, and `Prob (F-statistic)` are explained in the table below. `Log-Likelihood`, `AIC`, and `BIC` are for comparing models; we do not need them.
- **Middle block: one row per coefficient**, from $\beta_0$ (`const`) to $\beta_9$ (`Am. Indian/AK Native`). `coef` is the estimate $\hat\beta_j$, `std err` its standard error, `t` the t value, `P>|t|` the p value, and `[0.025` and `0.975]` the two ends of the 95% confidence interval.
- **Bottom block** (`Omnibus`, `Durbin-Watson`, `Jarque-Bera`, `Skew`, `Kurtosis`, `Cond. No.`): checks of the assumptions about the errors. We do not use them in this course.

| Statistic | Formula | In plain words |
|---|---|---|
| standard error | $SE(\hat\beta_j)$, computed from the residuals | how much $\hat\beta_j$ would vary from one sample of essays to another |
| t value | $t_j = \hat\beta_j \,/\, SE(\hat\beta_j)$ | how many standard errors the estimate lies away from 0 |
| p value | the chance of a t value at least this far from 0 if the true $\beta_j$ were 0 | the test of $H_0: \beta_j = 0$. When $p < 0.05$, the coefficient is called *statistically significant* at the 0.05 level |
| 95% confidence interval | about $\hat\beta_j \pm 1.96 \times SE(\hat\beta_j)$ | a range of plausible values for $\beta_j$. It contains 0 exactly when $p \ge 0.05$ |
| $R^2$ | $1 - \dfrac{\sum_i (\text{score}_i - \widehat{\text{score}}_i)^2}{\sum_i (\text{score}_i - \overline{\text{score}})^2}$ | the share of the variation in scores that the model accounts for: 0 = none, 1 = all |
| adjusted $R^2$ | $1 - (1 - R^2)\,\dfrac{n - 1}{n - k - 1}$ | $R^2$ with a small penalty for every predictor |
| F statistic | compares this model with one that has only $\beta_0$ | its p value, `Prob (F-statistic)`, tests $H_0: \beta_1 = \dots = \beta_9 = 0$, that none of the predictors helps |

A p value printed as `0.000` is not zero, only smaller than 0.0005. Report it as *p* < 0.001.

#### 🗣️ Your interpretation (optional)

**This is optional. You may use AI to help interpret the results, but AI can be wrong while sounding convincing, so check what it tells you against your own output.**

Please write a few sentences about the model's results, starting with the R² and the F test. Then describe the coefficients that are statistically significant at the 0.05 level, and what they say about the association of each predictor with score. You do not need to interpret the intercept or any coefficient that is not significant.

In [ ]:
# Use comment or print to write your interpretation here:


---
**Data credit.** Essays and scores come from the ASAP 2.0 corpus (GitHub: `scrosseye/ASAP_2.0`), shared under [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/). Crossley, S. (2025). *A large-scale corpus for assessing source-based writing quality: ASAP 2.0.* Zenodo. https://doi.org/10.5281/zenodo.14781349. This assignment uses the 3,500 essays written to the "Driverless cars" prompt. The only changes: unused columns were removed and the text column was moved to the end.